<a href="https://colab.research.google.com/github/ramanji567/TrendPulse_ramanji/blob/main/task1_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import requests
import time
import json
from datetime import datetime

headers = {
    "User-Agent":"TrendPluse/1.0"
}
# Corrected: Assign data directly to data_1
categories = {
    "technology":["AI","software","tech","code","computer","data","cloud","API","GPU","LLM"],
    "worldnews":["war","government","country","president","election","climate","attack","global"],
    "sports":["NFL","NBA","FIFA","sport","game","team","player","league","championship"],
    "science":["research","study","space","physics","biology","discovery","NASA","genome"],
    "entertainment":["movie","film","music","Netflix","game","book","show","award","streaming"]

}
def get_category (title):
  title = title.lower()
  for category_name,keywords in categories.items(): # Renamed 'keyword' to 'category_name' to avoid conflict
    for word in keywords:
      if word in title:
        return category_name # Return category_name here
  return "other"

def get_story_id():
  try:
    url = f"https://hacker-news.firebaseio.com/v0/topstories.json"
    response = requests.get(url,headers = headers)
    return response.json()
    # print(data) # Removed unreachable and undefined 'print(data)'
  except:
    print("failed to fetch")
    return [] # Return an empty list on failure

def get_story(story_id):

   try:
      url = f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json" # Corrected to use the passed story_id
      response = requests.get(url,headers=headers)
      return response.json()
   except:
     print(f"failed to fetch story {story_id}") # Added story_id to error message
     return None # Return None on failure
def main():
  story_ids = get_story_id()

  result = {
      "technology":[],
      "worldnews":[],
      "sports":[],
      "science":[],
      "entertainment":[]
  }

  for category in result:
    count =0

    for story_id in story_ids:
      if count >=25:
        break

      story = get_story(story_id)
      if not story or "title" not in story:
        continue
      category_name = get_category(story["title"]) # Corrected to call get_category with the story title
      if category_name == category:
        data ={
            "post_id":story.get("id"),
            "title":story.get("title"),
            "category":category_name,
            "score":story.get("score"),
            "num_comments":story.get("descendants"),
            "author":story.get("by"),
            "collected_at":datetime.now().isoformat()
        }

        result[category].append(data)
        count += 1 # Increment count when a story is added to a category
    #time.sleep(1) # Reduced sleep time to 1 second for faster execution

  if not os.path.exists("data"):
     os.makedirs("data")
  filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

  with open(filename,"w")as f:
      json.dump(result,f,indent=2)
  print(f"saved file: {filename}") # Corrected print statement

if __name__=="__main__":
  main()

saved file: data/trends_20260409.json
